# Creating a weather map

In [144]:
import os 
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")

In [145]:
# Get the model 
from langchain.chat_models import init_chat_model
llm = init_chat_model(model="llama-3.3-70b-versatile", model_provider="Groq")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x12686e030>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x12686e7b0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [146]:
# invoke the model 
llm.invoke("What is the weather in Miami, USA today?")

AIMessage(content='I\'m an AI, I don\'t have real-time access to current weather conditions. But I can suggest some ways for you to find out the current weather in Miami, USA.\n\nYou can check the weather on various websites, such as:\n\n1. National Weather Service (NWS): [www.weather.gov](http://www.weather.gov)\n2. AccuWeather: [www.accuweather.com](http://www.accuweather.com)\n3. Weather.com: [www.weather.com](http://www.weather.com)\n4. Google: Simply type "Miami weather" in the search bar, and you\'ll get the current weather conditions.\n\nYou can also check the weather on your smartphone by downloading a weather app, such as Dark Sky or Weather Underground.\n\nPlease note that the weather can change rapidly, so it\'s always a good idea to check the forecast frequently for the most up-to-date information.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 182, 'prompt_tokens': 45, 'total_tokens': 227, 'completion_time': 0.418546509, 'completion_tokens_

## Create a tool 

In [ ]:
from langchain.tools import tool 
import requests

@tool
def getWeather(location:str,zipcode:str)->str:
      """ return weather information based on the given location information
      args: city and or zip code 

      return: json format weather information
      """
      url = "https://api.openweathermap.org/data/2.5/weather"

      params={
            "q":"location",
            "zip":zipcode,
            "appid":OPENWEATHER_API_KEY,
            "units":"metric" #imperial
      }

      try:
            response = requests.get(url,params,timeout=10)
            data = response.json()
            return data
      except:
            return f"Weather information not found"

In [148]:
llm_with_tools = llm.bind_tools([getWeather])

In [149]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage
messages = [
    SystemMessage(content="you are an helpful weather assistant, which can provide realtime weather information using tools call")
]

In [150]:

## Add the prompt
# messages = []
messages.append(HumanMessage(content="What is the weather in Peoria with zipcode 61525 today, make sure you mention high and low temperature?"))

In [151]:
aiMessage = llm_with_tools.invoke(messages)
messages.append(aiMessage)

In [152]:
aiMessage.tool_calls

[{'name': 'getWeather',
  'args': {'location': 'Peoria', 'zipcode': '61525'},
  'id': 'vk4c1eczx',
  'type': 'tool_call'}]

In [153]:
toolMessage = getWeather.invoke(aiMessage.tool_calls[0])

In [154]:
toolMessage

ToolMessage(content='{"coord": {"lon": -79.6371, "lat": 39.2139}, "weather": [{"id": 801, "main": "Clouds", "description": "few clouds", "icon": "02d"}], "base": "stations", "main": {"temp": 32.57, "feels_like": 36.27, "temp_min": 30.26, "temp_max": 32.72, "pressure": 1018, "humidity": 53, "sea_level": 1018, "grnd_level": 937}, "visibility": 10000, "wind": {"speed": 2.91, "deg": 287, "gust": 4.14}, "clouds": {"all": 17}, "dt": 1783114730, "sys": {"type": 2, "id": 2076251, "country": "US", "sunrise": 1783072607, "sunset": 1783126107}, "timezone": -14400, "id": 4811007, "name": "Location", "cod": 200}', name='getWeather', tool_call_id='vk4c1eczx')

In [155]:
messages.append(toolMessage) 
llm_with_tools.invoke(messages).content

'The current weather in Peoria with zipcode 61525 is few clouds with a temperature of 32.57°F. The high temperature today is expected to be around 32.72°F, while the low temperature is expected to be around 30.26°F.'

In [156]:
#structured output  
from pydantic import BaseModel

class StructuredWeather(BaseModel):
    location:str
    high_temperature:float
    low_temperature:float

In [157]:
llm_with_structured_op= llm.with_structured_output(StructuredWeather)
llm_with_structured_op.invoke(messages)

StructuredWeather(location='Peoria, 61525', high_temperature=32.72, low_temperature=30.26)